In [46]:

import os
import random
import re
from collections import defaultdict
from xml.etree.ElementTree import tostring

from PIL import Image
import numpy as np
from pathlib import Path

class DatasetGenerator:
    def __init__(self, background_dir, ground_units_dir, air_units_dir, towers_dir, output_dir):
        """
        Inizializza il generatore di dataset

        Args:
            background_dir: cartella con le immagini di background
            ground_units_dir: cartella con PNG delle truppe di terra
            air_units_dir: cartella con PNG delle truppe aeree
            towers_dir: cartella con PNG delle torri
            output_dir: cartella di output per immagini e labels
        """
        self.background_dir = Path(background_dir)
        self.ground_units_dir = Path(ground_units_dir)
        self.air_units_dir = Path(air_units_dir)
        self.towers_dir = Path(towers_dir)
        self.output_dir = Path(output_dir)
        # Crea le cartelle di output
        (self.output_dir / "images").mkdir(parents=True, exist_ok=True)
        (self.output_dir / "labels").mkdir(parents=True, exist_ok=True)

        # Carica le liste dei file
        self.backgrounds = list(self.background_dir.glob("*.jpg")) + list(self.background_dir.glob("*.png"))
        self.ground_units = list(self.ground_units_dir.glob("*.png"))
        self.air_units = list(self.air_units_dir.glob("*.png"))
        self.towers = list(self.towers_dir.glob("*.png"))
        nomi = []
        for x in self.ground_units:
            nomi.append(os.path.basename(x))
        buckets = defaultdict(list)
        for f in nomi:
            # prende la parte tra _ e ( oppure _ e .png
            match = re.search(r'_(.*?)(?:\s*\(|\.png)', f)
            if match:
                key = match.group(1)
                buckets[key].append(f)

        # Mappa delle classi (modifica secondo le tue esigenze)
        self.class_map = {}
        n=0
        for k in buckets.keys():
            self.class_map[k] = n
            n += 1
        self.class_map["tower"] = n

    def load_png_with_alpha(self, path):
        """Carica un PNG preservando il canale alpha"""
        print(path)
        img = Image.open(path).convert("RGBA")
        nome = re.search(r'_(.*?)(?:\s*\(|\.png)', os.path.basename(path))
        print(nome)
        return img , nome

    def paste_with_transparency(self, background, overlay, position):
        """Incolla un'immagine con trasparenza su un background"""
        background.paste(overlay, position, overlay)
        return background

    def get_bbox_yolo(self, x, y, width, height, img_width, img_height):
        """
        Converte coordinate bbox in formato YOLO
        YOLO formato: [class_id, x_center, y_center, width, height] (normalizzato 0-1)
        """
        x_center = (x + width / 2) / img_width
        y_center = (y + height / 2) / img_height
        norm_width = width / img_width
        norm_height = height / img_height
        return [x_center, y_center, norm_width, norm_height]

    def generate_image(self, num_ground_units, num_towers, num_air_units):
        """
        Genera una singola immagine con annotazioni YOLO

        Returns:
            image: immagine PIL generata
            annotations: lista di annotazioni YOLO [class_id, x_center, y_center, width, height]
        """
        # Carica background casuale
        bg_path = random.choice(self.backgrounds)
        background = Image.open(bg_path).convert("RGBA")
        img_width, img_height = background.size

        annotations = []

        # Livello 2: Truppe di terra e torri
        # Aggiungi torri
        x = 50
        y = 460
        ally_princess_tower, a = self.load_png_with_alpha(self.towers[1])
        ally_king_tower, a = self.load_png_with_alpha(self.towers[0])
        enemy_princess_tower, a = self.load_png_with_alpha(self.towers[3])
        enemy_king_tower, a = self.load_png_with_alpha(self.towers[2])
        for _ in range(2):
            if not self.towers:
                continue

            tw, th = ally_princess_tower.size
            background = self.paste_with_transparency(background, ally_princess_tower, (x, y))
            # Calcola bbox YOLO
            bbox = self.get_bbox_yolo(x, y, tw, th, img_width, img_height)
            annotations.append([self.class_map['tower']] + bbox)
            x=315
        x = 55
        y = 100
        for _ in range(2):
            if not self.towers:
                continue

            tw, th = enemy_princess_tower.size
            background = self.paste_with_transparency(background, enemy_princess_tower, (x, y))
            # Calcola bbox YOLO
            bbox = self.get_bbox_yolo(x, y, tw, th, img_width, img_height)
            annotations.append([self.class_map['tower']] + bbox)
            x=320
        x=180
        y=20
        tw, th = enemy_king_tower.size
        background = self.paste_with_transparency(background, enemy_king_tower, (x, y))
        # Calcola bbox YOLO
        bbox = self.get_bbox_yolo(x, y, tw, th, img_width, img_height)
        annotations.append([self.class_map['tower']] + bbox)
        y=525
        tw, th = ally_king_tower.size
        background = self.paste_with_transparency(background, ally_king_tower, (x, y))
        # Calcola bbox YOLO
        bbox = self.get_bbox_yolo(x, y, tw, th, img_width, img_height)
        annotations.append([self.class_map["tower"]] + bbox)
        # Aggiungi truppe di terra
        for _ in range(num_ground_units):
            if not self.ground_units:
                continue
            unit , nome = self.load_png_with_alpha(random.choice(self.ground_units))
            uw, uh = unit.size
            x = random.randint(0, max(0, img_width - uw))
            y = random.randint(0, max(0, img_height - uh))

            background = self.paste_with_transparency(background, unit, (x, y))

            bbox = self.get_bbox_yolo(x, y, uw, uh, img_width, img_height)
            annotations.append([self.class_map[nome]] + bbox)

        # Livello 3: Truppe aeree
        for _ in range(num_air_units):
            if not self.air_units:
                continue
            air, nome = self.load_png_with_alpha(random.choice(self.air_units))
            aw, ah = air.size

            x = random.randint(0, max(0, img_width - aw))
            y = random.randint(0, max(0, img_height - ah))

            background = self.paste_with_transparency(background, air, (x, y))

            bbox = self.get_bbox_yolo(x, y, aw, ah, img_width, img_height)
            annotations.append([self.class_map['giant']] + bbox)

        return background, annotations

    def generate_dataset(self, num_images, distribution=None):
        """
        Genera un dataset bilanciato

        Args:
            num_images: numero totale di immagini da generare
            distribution: dizionario con la distribuzione desiderata per ogni configurazione
                         Se None, usa distribuzione uniforme bilanciata
        """
        if distribution is None:
            # Distribuzione bilanciata di default
            # Varia il numero di oggetti per classe
            distribution = [
                {'ground': (1, 3), 'tower': (1, 2), 'air': (0, 2)},  # Prevalenza terra
                {'ground': (0, 2), 'tower': (2, 4), 'air': (0, 1)},  # Prevalenza torri
                {'ground': (0, 1), 'tower': (1, 2), 'air': (2, 4)},  # Prevalenza aeree
                {'ground': (2, 4), 'tower': (2, 3), 'air': (2, 3)},  # Misto bilanciato
            ]

        for i in range(num_images):
            # Scegli una configurazione casuale
            config = random.choice(distribution)

            num_ground = random.randint(*config['ground'])
            num_towers = random.randint(*config['tower'])
            num_air = random.randint(*config['air'])

            # Genera immagine
            image, annotations = self.generate_image(num_ground, num_towers, num_air)

            # Salva immagine
            img_name = f"image_{i:05d}.jpg"
            image_path = self.output_dir / "images" / img_name
            image.convert("RGB").save(image_path, "JPEG")

            # Salva annotazioni YOLO
            label_name = f"image_{i:05d}.txt"
            label_path = self.output_dir / "labels" / label_name

            with open(label_path, 'w') as f:
                for ann in annotations:
                    class_id = ann[0]
                    bbox = ann[1:]
                    f.write(f"{class_id} {' '.join(map(str, bbox))}\n")

            if (i + 1) % 100 == 0:
                print(f"Generate {i + 1}/{num_images} immagini")

        print(f"Dataset completato! {num_images} immagini generate.")
        self.create_yaml_config()

    def create_yaml_config(self):
        """Crea il file di configurazione YAML per YOLO"""
        print(self.class_map)
        yaml_content = f"""# Dataset configuration for YOLO
path: {self.output_dir.absolute()}
train: images
val: images

# Classes
names:
    {self.class_map}
nc: {len(self.class_map)}  # number of classes
"""
        yaml_path = self.output_dir / "dataset.yaml"
        with open(yaml_path, 'w') as f:
            f.write(yaml_content)
        print(f"File di configurazione salvato in: {yaml_path}")


# Funzione di test e verifica
def check_directories(bg_dir, ground_dir, air_dir, tower_dir):
    """Verifica che le cartelle esistano e contengano file"""
    print("=== VERIFICA CARTELLE ===")

    dirs = {
        'Backgrounds': bg_dir,
        'Ground Units': ground_dir,
        'Air Units': air_dir,
        'Towers': tower_dir
    }

    all_ok = True
    for name, path in dirs.items():
        p = Path(path)
        if not p.exists():
            print(f"❌ {name}: cartella '{path}' NON ESISTE")
            all_ok = False
        else:
            files = list(p.glob("*.png")) + list(p.glob("*.jpg"))
            print(f"✓ {name}: {len(files)} file trovati in '{path}'")
            if len(files) == 0:
                all_ok = False

    return all_ok


In [47]:
# Esempio di utilizzo
if __name__ == "__main__":
    generator = DatasetGenerator(
        background_dir="segments/backgrounds",      # Cartella con i background
        ground_units_dir="segments/ground_units",   # Cartella con truppe di terra PNG
        air_units_dir="segments/air_units",         # Cartella con truppe aeree PNG
        towers_dir="segments/towers",               # Cartella con torri PNG
        output_dir="dataset_output"        # Cartella di output
    )

    # Genera 1000 immagini bilanciate
    generator.generate_dataset(num_images=10)

    # Oppure con distribuzione personalizzata
    # custom_dist = [
    #     {'ground': (2, 5), 'tower': (1, 3), 'air': (0, 2)},
    #     {'ground': (0, 2), 'tower': (3, 5), 'air': (1, 3)},
    # ]
    # generator.generate_dataset(num_images=1000, distribution=custom_dist)

segments\towers\ally_princess_tower.png
<re.Match object; span=(4, 23), match='_princess_tower.png'>
segments\towers\ally_king_tower.png
<re.Match object; span=(4, 19), match='_king_tower.png'>
segments\towers\enemy_princess_tower.png
<re.Match object; span=(5, 24), match='_princess_tower.png'>
segments\towers\enemy_king_tower.png
<re.Match object; span=(5, 20), match='_king_tower.png'>
segments\air_units\ally_minion.png
<re.Match object; span=(4, 15), match='_minion.png'>
segments\towers\ally_princess_tower.png
<re.Match object; span=(4, 23), match='_princess_tower.png'>
segments\towers\ally_king_tower.png
<re.Match object; span=(4, 19), match='_king_tower.png'>
segments\towers\enemy_princess_tower.png
<re.Match object; span=(5, 24), match='_princess_tower.png'>
segments\towers\enemy_king_tower.png
<re.Match object; span=(5, 20), match='_king_tower.png'>
segments\ground_units\ally_giant.png
<re.Match object; span=(4, 14), match='_giant.png'>


KeyError: <re.Match object; span=(4, 14), match='_giant.png'>